# MedAI Training Pipeline

Train custom BiomedBERT with dual heads on Kaggle P100 GPU.

In [ ]:
!pip install -q torch transformers accelerate sentence-transformers faiss-cpu scikit-learn tqdm datasets seqeval

In [ ]:
import json
import os
import torch
import transformers
import datasets
from pathlib import Path
from pprint import pprint

print(f"Torch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
import os
os.makedirs("/kaggle/working/models", exist_ok=True)
os.makedirs("/kaggle/working/data", exist_ok=True)
print("Kaggle working directories are ready.")

In [ ]:
instructions = """
Upload the project files before running training:
1. In Kaggle, open the right sidebar and add src/model.py and src/train.py as dataset files or upload them into /kaggle/working/src/.
2. Make sure /kaggle/working/src/__init__.py exists so Python can import the package.
3. The training script will save model outputs under /kaggle/working/models/.
"""
os.makedirs("/kaggle/working/src", exist_ok=True)
Path("/kaggle/working/src/__init__.py").touch(exist_ok=True)
print(instructions)

In [ ]:
from src.train import main
main()

In [ ]:
from transformers import AutoTokenizer
from src.model import MedicalReportModel

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = "/kaggle/working/models/best_model.pt"
tokenizer_path = "/kaggle/working/models/tokenizer"
sample_text = "Patient has HbA1c 9.1% and WBC 14000/uL."

model = MedicalReportModel().to(device)
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
prediction = model.predict(sample_text, tokenizer, device)
pprint(prediction)

## Download Outputs

Download `/kaggle/working/models/` after training. It contains `best_model.pt`, the tokenizer folder, and the label maps needed for inference.